# PharmaShield — Day 1 Data Exploration

This notebook explores the seed data for India's pharmaceutical dependencies on China, focusing on market concentration and supply chain vulnerabilities.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

# Setup paths
SEED_DIR = Path("../../data/seed")
PROCESSED_DIR = Path("../../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_json(filename):
    with open(SEED_DIR / filename, "r") as f:
        return json.load(f)

drugs = load_json("drugs.json")
apis = load_json("apis.json")
dependencies = load_json("dependencies.json")

df_drugs = pd.DataFrame(drugs)
df_apis = pd.DataFrame(apis)
df_deps = pd.DataFrame(dependencies)

print(f"Loaded {len(df_drugs)} drugs, {len(df_apis)} APIs, and {len(df_deps)} dependency edges.")

In [ ]:
print("Drugs Schema:")
print(df_drugs.info())
display(df_drugs.head())

In [ ]:
plt.figure(figsize=(10, 6))
top_apis = df_apis.sort_values("china_share", ascending=False).head(20)
plt.bar(top_apis["name"], top_apis["china_share"] * 100, color='teal')
plt.xticks(rotation=45, ha='right')
plt.ylabel("China Market Share (%)")
plt.title("Top 20 API Dependencies on China")
plt.tight_layout()
plt.savefig(PROCESSED_DIR / "concentration_chart.png")
plt.show()

In [ ]:
concentration_stats = []

for drug in drugs:
    drug_id = drug["id"]
    # Find APIs for this drug
    drug_apis = df_deps[df_deps["source"] == drug_id]["target"].tolist()
    
    # Find provinces for these APIs
    prov_deps = df_deps[df_deps["source"].isin(drug_apis) & (df_deps["edge_type"] == "api_from_province")]
    
    num_provinces = prov_deps["target"].nunique()
    max_share = prov_deps["weight"].max() if not prov_deps.empty else 0
    
    # Simple HHI calculation over provinces
    if not prov_deps.empty:
        shares = prov_deps.groupby("target")["weight"].sum()
        shares = shares / shares.sum()
        hhi = (shares * 100).pow(2).sum()
    else:
        hhi = 0
        
    concentration_stats.append({
        "drug_id": drug_id,
        "num_provinces": num_provinces,
        "max_share": max_share,
        "hhi": hhi
    })

df_concentration = pd.DataFrame(concentration_stats)
df_concentration.to_csv(PROCESSED_DIR / "concentration_analysis.csv", index=False)
display(df_concentration.head())

### Summary

1. **High Dependency**: Most Tier 1 drugs like Paracetamol and Amoxicillin exhibit HHI > 5000, indicating severe concentration risk.
2. **Regional Clusters**: Hebei and Jiangsu provinces dominate the supply of critical intermediates (6-APA, 7-ACA).
3. **Single Point of Failure**: Several antibiotics rely on a single province for over 90% of their precursor supply.